# FDA FAERS Post-Marketing Pharmacovigilance Surveillance: GLP-1 Receptor Agonists Safety Profile

## 1. Executive Summary & Clinical Context
The rapid global adoption of glucagon-like peptide-1 receptor agonists (GLP-1 RAs) and dual GIP/GLP-1 receptor agonists—such as Semaglutide and Tirzepatide—for Type 2 Diabetes (T2D) and chronic weight management has led to an exponential increase in real-world exposure.

While pre-approval randomized clinical trials (RCTs) establish basic safety and efficacy, post-marketing spontaneous reporting systems are essential for detecting rare, latent, or severe adverse drug reactions (ADRs) across heterogeneous, unmonitored patient populations.

## 2. Project Objectives
* **Primary Objective:** Build an auditable, reproducible end-to-end data pipeline to systematically isolate, clean, and profile GLP-1 receptor agonist safety reports from the FDA Adverse Event Reporting System (FAERS).
* **Secondary Objectives:**
  * Quantify report volumes stratified by pharmacological active ingredient and commercial brand/indication.
  * Audit data fragmentation, reporting synonyms, and the emergence of non-standard compounded formulations.
  * Characterize severe clinical outcomes (death, hospitalization, life-threatening events) across distinct therapeutic cohorts.
  * Track and document all data exclusion steps with strict regulatory transparency (e.g., handling official FDA retraction logs and potential confounders).

## 3. Data Source Architecture & Regulatory Standards
* **Dataset:** FDA FAERS ASCII quarterly extract (`2026Q2`).
* **Relational Schema Utilized:**
  * `DEMO26Q2`: Demographic and case-level metadata (Patient identifier, age, sex, reporter country).
  * `DRUG26Q2`: Medication exposure details (Drug name, active ingredient, role code, route of administration).
  * `REAC26Q2`: Reported adverse events coded in Medical Dictionary for Regulatory Activities (MedDRA) Preferred Terms (PT).
  * `OUTC26Q2`: Patient clinical outcomes (Death, Hospitalization, Disability, Life-Threatening).
  * `DELETE26Q2`: Official FDA case retraction and deduplication log.
* **Methodological Standards:** Adheres to ICH E2B guidelines for pharmacovigilance data management and ICH E2D post-approval safety data principles.

---

## 4. Phase 1: Environment Setup & Raw Exposure Screening

### Methodological Rationale for Query 01:
To begin the analysis without introducing premature aggregation bias, we first execute an exploratory screening query against the raw `DRUG26Q2` table.

1. **Regulatory Case Deduplication:** Before evaluating drug exposure, cases flagged in the official FDA retraction log (`DELETE26Q2.txt`) are isolated and excluded. Retracted cases represent administratively revoked, merged, or duplicated entries that must be purged to prevent frequency distortion.
2. **Causal Attribution Filter (`ROLE_COD = 'PS'`):** In spontaneous safety databases, patients frequently take multiple concomitant medications. Restricting the cohort to `ROLE_COD = 'PS'` (Primary Suspect) isolates cases where the treating healthcare professional or reporter identified the GLP-1 therapy as the primary suspect agent driving the adverse event, eliminating background noise from secondary or concomitant medications.
3. **Broad Substring Retrieval:** Search patterns evaluate both declared active ingredients (`prod_ai`) and trade names (`drugname`) across all known GLP-1 class agents (Semaglutide, Tirzepatide, Liraglutide, Dulaglutide, Exenatide, Lixisenatide) to identify the raw spectrum of commercial reporting nomenclature.
4. **Unique Case Accounting:** Aggregation uses `COUNT(DISTINCT primaryid)` to ensure each clinical case is counted exactly once, regardless of whether multiple dose lines or pen devices were logged for the same patient event.

In [ ]:
"""
FDA FAERS Pharmacovigilance Analytics Pipeline
Module: 01_raw_entity_exploration.py
Author: Dalina Gaxiola Báez
Description:
    Performs raw exploratory screening of all reported brand names and declared
    active ingredients within the GLP-1 receptor agonist therapeutic class.
    Executes deduplication against official FDA retraction logs (DELETE26Q2.txt)
    and restricts retrieval to Primary Suspect agents (ROLE_COD = 'PS').
"""

import os
import duckdb
import pandas as pd

# ==========================================
# 1. ENVIRONMENT & PATH CONFIGURATION
# ==========================================
DATA_DIR = "/content/drive/MyDrive/TripleTen Projects/Proyectos/FAERS"
DELETE_FILE = os.path.join(DATA_DIR, "DELETE26Q2.txt")
DRUG_FILE = os.path.join(DATA_DIR, "DRUG26Q2.txt")

# ==========================================
# 2. RAW AUDIT EXTRACTION QUERY
# ==========================================
query_audit = f"""
WITH
-- Purge officially retracted cases per FDA maintenance logs
retracted_cases AS (
    SELECT DISTINCT column0::VARCHAR AS caseid
    FROM read_csv('{DELETE_FILE}', header=False)
),

-- Extract raw Primary Suspect GLP-1 records
raw_glp1_mentions AS (
    SELECT DISTINCT
        d.primaryid::VARCHAR AS primaryid,
        UPPER(TRIM(d.drugname)) AS reported_drugname,
        UPPER(TRIM(d.prod_ai)) AS reported_prod_ai
    FROM read_csv('{DRUG_FILE}', delim='$', header=True) d
    WHERE d.role_cod = 'PS'
      AND (
          -- Active ingredient screening
          LOWER(d.prod_ai) LIKE '%semaglutide%'
          OR LOWER(d.prod_ai) LIKE '%tirzepatide%'
          OR LOWER(d.prod_ai) LIKE '%liraglutide%'
          OR LOWER(d.prod_ai) LIKE '%dulaglutide%'
          OR LOWER(d.prod_ai) LIKE '%exenatide%'
          OR LOWER(d.prod_ai) LIKE '%lixisenatide%'
          -- Brand / Trade name screening
          OR LOWER(d.drugname) LIKE '%ozempic%'
          OR LOWER(d.drugname) LIKE '%wegovy%'
          OR LOWER(d.drugname) LIKE '%mounjaro%'
          OR LOWER(d.drugname) LIKE '%zepbound%'
          OR LOWER(d.drugname) LIKE '%rybelsus%'
          OR LOWER(d.drugname) LIKE '%victoza%'
          OR LOWER(d.drugname) LIKE '%saxenda%'
          OR LOWER(d.drugname) LIKE '%trulicity%'
          OR LOWER(d.drugname) LIKE '%byetta%'
          OR LOWER(d.drugname) LIKE '%bydureon%'
          OR LOWER(d.drugname) LIKE '%soliqua%'
          OR LOWER(d.drugname) LIKE '%xultophy%'
      )
      AND d.caseid::VARCHAR NOT IN (SELECT caseid FROM retracted_cases)
)

-- Frequency aggregation of unique cases by reported terminology
SELECT
    reported_drugname,
    reported_prod_ai,
    COUNT(DISTINCT primaryid) AS total_cases
FROM raw_glp1_mentions
GROUP BY reported_drugname, reported_prod_ai
ORDER BY total_cases DESC;
"""

# ==========================================
# 3. PIPELINE EXECUTION
# ==========================================
con = duckdb.connect()
df_raw_audit = con.execute(query_audit).df()

# Configure Pandas display to review complete findings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
display(df_raw_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,reported_drugname,reported_prod_ai,total_cases
0,ZEPBOUND,TIRZEPATIDE,11224
1,OZEMPIC,SEMAGLUTIDE,8170
2,WEGOVY,SEMAGLUTIDE,6556
3,MOUNJARO,TIRZEPATIDE,5200
4,ZEPBOUND KWIKPEN,TIRZEPATIDE,2252
5,TRULICITY,DULAGLUTIDE,969
6,SEMAGLUTIDE,SEMAGLUTIDE,584
7,RYBELSUS,SEMAGLUTIDE,439
8,TIRZEPATIDE,TIRZEPATIDE,320
9,LIRAGLUTIDE,LIRAGLUTIDE,187


## 2. Clinical Data Audit & Exclusion Methodology

### 2.1 Audit Findings on Raw Reported Entities
The initial exploratory query (`01_raw_entity_exploration.py`) identified 36 unique drug reporting combinations (`reported_drugname` + `reported_prod_ai`) in the 2026 Q2 extract representing post-marketing surveillance where a GLP-1 agent was designated as the Primary Suspect (`ROLE_COD = 'PS'`).

A structured clinical evaluation revealed three distinct exposure categories:

| Exposure Category | Formulations Identified | Total Unique Cases | Methodological Action | Clinical Justification |
|---|---|:---:|:---:|---|
| **FDA-Approved Monotherapies** | *Zepbound, Ozempic, Wegovy, Mounjaro, Trulicity, Rybelsus, Saxenda, Victoza, Byetta* (and generic INN entries) | **35,849** | **INCLUDED** | Represents clean, standardized exposure to target molecules. Delivery device variants (e.g., *Zepbound Kwikpen*) are harmonized into primary brand cohorts. |
| **Fixed-Ratio Insulin Combinations** | *Soliqua 100/33* (Insulin Glargine + Lixisenatide), *Xultophy 100/3.6* (Insulin Degludec + Liraglutide) | **160** | **EXCLUDED** | **Glycemic Confounder:** Fixed-dose basal insulin introduces direct pharmacological confounding for severe metabolic outcomes (e.g., severe hypoglycemia, coma). |
| **Unapproved Compounded Formulations** | Mixtures combining Tirzepatide/Semaglutide with *Cyanocobalamin (B12), Pyridoxine (B6), Glycine, Niacinamide, Levocarnitine, Inositol* | **65** | **EXCLUDED** | **Quality & Formulation Heterogeneity:** Unregulated compounding pharmacy mixtures lack uniform dosing consistency, bioavailability standards, and sterile controls, introducing uncontrolled variance into post-marketing safety signals. |

**Total Screened Case Universe:** 36,074 unique patient reports  
**Target Clean Monotherapy Cohort:** 35,849 unique patient reports (99.38% of total volume)  
**Total Excluded Cases:** 225 reports (0.62% of total volume: 160 insulin combinations + 65 compounded mixtures)

---

### 2.2 Next Step: Protocol-Compliant Cohort Harmonization & Triage (Query 02)
In the next execution block, we apply SQL rules to:
1. Filter out all 225 excluded records flagged as confounded combinations or non-standard compounded preparations.
2. Unify device-specific variants (`ZEPBOUND KWIKPEN` → `ZEPBOUND`, `MOUNJARO KWIKPEN` → `MOUNJARO`).
3. Stratify the final clean monotherapy cohort by commercial brand and primary therapeutic indication (*Obesity* vs. *Type 2 Diabetes* vs. *Unspecified INN*) to prepare the base for clinical severity outcomes analysis (`OUTC26Q2`).

In [ ]:
"""
FDA FAERS Pharmacovigilance Analytics Pipeline
Module: 02_glp1_clean_cohort_stratification.py
Author: Dalina Gaxiola Báez
Description:
    Constructs the protocol-compliant GLP-1 receptor agonist safety cohort.
    Enforces clinical exclusion criteria (insulin combinations and compounded mixtures),
    consolidates proprietary delivery device nomenclature, and stratifies cases by
    commercial brand and primary clinical indication (Type 2 Diabetes vs. Obesity).
"""

# Nota el prefijo rf""" para evitar la alerta de secuencia de escape
query_clean_cohort = rf"""
WITH
-- 1. Purge officially retracted FAERS records
retracted_cases AS (
    SELECT DISTINCT column0::VARCHAR AS caseid
    FROM read_csv('{DELETE_FILE}', header=False)
),

-- 2. Triage exposure records and apply exclusion filters
classified_exposure AS (
    SELECT DISTINCT
        d.primaryid::VARCHAR AS primaryid,
        d.caseid::VARCHAR AS caseid,
        UPPER(TRIM(d.drugname)) AS reported_drugname,
        UPPER(TRIM(d.prod_ai)) AS reported_prod_ai,

        -- Standardize brand and target indication
        CASE
            -- Tirzepatide cohort
            WHEN UPPER(d.drugname) LIKE '%ZEPBOUND%' THEN 'ZEPBOUND (Tirzepatide - Obesity)'
            WHEN UPPER(d.drugname) LIKE '%MOUNJARO%' THEN 'MOUNJARO (Tirzepatide - T2D)'
            WHEN (UPPER(d.drugname) = 'TIRZEPATIDE' OR UPPER(d.prod_ai) = 'TIRZEPATIDE')
                 AND UPPER(d.drugname) NOT LIKE '%CYANOCOBALAMIN%'
                 AND UPPER(d.drugname) NOT LIKE '%PYRIDOXINE%'
                 AND UPPER(d.drugname) NOT LIKE '%GLYCINE%'
                 AND UPPER(d.drugname) NOT LIKE '%NIACINAMIDE%'
                 AND UPPER(d.drugname) NOT LIKE '%LEVOCARNITINE%'
                 AND UPPER(d.drugname) NOT LIKE '%METHYLCOBALAMIN%'
                 THEN 'TIRZEPATIDE (Generic/Unspecified)'

            -- Semaglutide cohort
            WHEN UPPER(d.drugname) LIKE '%WEGOVY%' THEN 'WEGOVY (Semaglutide - Obesity)'
            WHEN UPPER(d.drugname) LIKE '%OZEMPIC%' THEN 'OZEMPIC (Semaglutide - T2D)'
            WHEN UPPER(d.drugname) LIKE '%RYBELSUS%' THEN 'RYBELSUS (Semaglutide - Oral T2D)'
            WHEN (UPPER(d.drugname) = 'SEMAGLUTIDE' OR UPPER(d.prod_ai) = 'SEMAGLUTIDE')
                 AND UPPER(d.drugname) NOT LIKE '%CYANOCOBALAMIN%'
                 AND UPPER(d.drugname) NOT LIKE '%PYRIDOXINE%'
                 AND UPPER(d.drugname) NOT LIKE '%GLYCINE%'
                 AND UPPER(d.drugname) NOT LIKE '%NIACINAMIDE%'
                 AND UPPER(d.drugname) NOT LIKE '%METHYLCOBALAMIN%'
                 THEN 'SEMAGLUTIDE (Generic/Unspecified)'

            -- Dulaglutide cohort
            WHEN UPPER(d.drugname) LIKE '%TRULICITY%' THEN 'TRULICITY (Dulaglutide - T2D)'
            WHEN UPPER(d.drugname) = 'DULAGLUTIDE' OR UPPER(d.prod_ai) = 'DULAGLUTIDE' THEN 'DULAGLUTIDE (Generic/Unspecified)'

            -- Liraglutide cohort
            WHEN UPPER(d.drugname) LIKE '%SAXENDA%' THEN 'SAXENDA (Liraglutide - Obesity)'
            WHEN UPPER(d.drugname) LIKE '%VICTOZA%' THEN 'VICTOZA (Liraglutide - T2D)'
            WHEN (UPPER(d.drugname) = 'LIRAGLUTIDE' OR UPPER(d.prod_ai) = 'LIRAGLUTIDE')
                 AND UPPER(d.drugname) NOT LIKE '%INSULIN%'
                 AND UPPER(d.prod_ai) NOT LIKE '%INSULIN%'
                 THEN 'LIRAGLUTIDE (Generic/Unspecified)'

            -- Exenatide cohort
            WHEN UPPER(d.drugname) LIKE '%BYETTA%' OR UPPER(d.drugname) LIKE '%BYDUREON%'
                 OR UPPER(d.drugname) = 'EXENATIDE' OR UPPER(d.prod_ai) = 'EXENATIDE'
                 THEN 'EXENATIDE (Generic/Brand)'

            ELSE 'OTHER_OR_EXCLUDED'
        END AS clinical_brand_cohort,

        -- Audit flag to track exclusion rationale
        CASE
            WHEN LOWER(d.drugname) LIKE '%soliqua%'
                 OR LOWER(d.drugname) LIKE '%xultophy%'
                 OR LOWER(d.prod_ai) LIKE '%insulin%'
                 THEN 'EXCLUDE_INSULIN_COMBINATION'
            WHEN LOWER(d.drugname) LIKE '%cyanocobalamin%'
                 OR LOWER(d.drugname) LIKE '%pyridoxine%'
                 OR LOWER(d.drugname) LIKE '%glycine%'
                 OR LOWER(d.drugname) LIKE '%niacinamide%'
                 OR LOWER(d.drugname) LIKE '%levocarnitine%'
                 OR LOWER(d.drugname) LIKE '%methylcobalamin%'
                 OR LOWER(d.prod_ai) LIKE '%\\%'
                 THEN 'EXCLUDE_COMPOUNDED'
            ELSE 'KEEP_MONOTHERAPY'
        END AS exclusion_flag

    FROM read_csv('{DRUG_FILE}', delim='$', header=True) d
    WHERE d.role_cod = 'PS'
      AND d.caseid::VARCHAR NOT IN (SELECT caseid FROM retracted_cases)
)

-- 3. Final aggregated monotherapy cohort table
SELECT
    clinical_brand_cohort,
    COUNT(DISTINCT primaryid) AS unique_cases,
    ROUND(COUNT(DISTINCT primaryid) * 100.0 / SUM(COUNT(DISTINCT primaryid)) OVER (), 2) AS pct_of_cohort
FROM classified_exposure
WHERE exclusion_flag = 'KEEP_MONOTHERAPY'
  AND clinical_brand_cohort != 'OTHER_OR_EXCLUDED'
GROUP BY clinical_brand_cohort
ORDER BY unique_cases DESC;
"""

# Pipeline execution
con = duckdb.connect()
df_clean_cohort = con.execute(query_clean_cohort).df()

# Display clean cohort breakdown
pd.set_option('display.max_rows', None)
display(df_clean_cohort)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clinical_brand_cohort,unique_cases,pct_of_cohort
0,ZEPBOUND (Tirzepatide - Obesity),13476,37.32
1,OZEMPIC (Semaglutide - T2D),8170,22.62
2,WEGOVY (Semaglutide - Obesity),6556,18.15
3,MOUNJARO (Tirzepatide - T2D),5202,14.40
4,TRULICITY (Dulaglutide - T2D),969,2.68
5,SEMAGLUTIDE (Generic/Unspecified),584,1.62
6,RYBELSUS (Semaglutide - Oral T2D),439,1.22
7,TIRZEPATIDE (Generic/Unspecified),320,0.89
8,LIRAGLUTIDE (Generic/Unspecified),187,0.52
9,SAXENDA (Liraglutide - Obesity),95,0.26


## 3. Clean Monotherapy Cohort Profile & Clinical Severity Triage

### 3.1 Quantitative Findings of the Harmonized Cohort
The execution of Query 02 established a protocol-compliant study cohort of **35,849 unique patient cases** exposed to FDA-approved GLP-1 receptor agonist monotherapies as Primary Suspect agents:

* **Therapeutic Domain Dominance:** Post-marketing reporting volume is heavily concentrated in contemporary dual GIP/GLP-1 and GLP-1 therapies (Tirzepatide and Semaglutide), which together account for **96.6%** of all reported monotherapy cases.
* **Indication Breakdown:**
  * **Chronic Weight Management (Obesity):** *Zepbound* (13,476 cases, 37.32%), *Wegovy* (6,556 cases, 18.15%), and *Saxenda* (95 cases, 0.26%) represent **55.73%** of the cohort volume.
  * **Type 2 Diabetes (T2D):** *Ozempic* (8,170 cases, 22.62%), *Mounjaro* (5,202 cases, 14.40%), *Trulicity* (969 cases, 2.68%), *Rybelsus* (439 cases, 1.22%), and *Victoza* (78 cases, 0.22%) represent **41.14%** of the cohort volume.
  * **Unspecified / Generic INN Reports:** 1,127 cases (3.13%) were submitted under generic active ingredient nomenclature without proprietary brand designation.

---

### 3.2 Next Step: Clinical Severity & Patient Outcome Profiling (Query 03)
Under international regulatory standards (**ICH E2D: Post-Approval Safety Data Management**), adverse drug reaction reports must be stratified by seriousness criteria. In spontaneous reporting systems, an adverse event is classified as serious if it results in:
* **DE:** Death
* **LT:** Life-Threatening experience
* **HO:** Inpatient hospitalization or prolongation of existing hospitalization
* **DS:** Persistent or significant disability/incapacity
* **CA:** Congenital anomaly/birth defect
* **RI:** Required intervention to prevent permanent impairment
* **OT:** Other medically important serious event

#### Analytical Objectives for Module 03 (`03_glp1_clinical_outcomes.py`):
1. **Relational Join with Outcome Data:** Link the clean monotherapy cohort (`primaryid`) against the FDA FAERS clinical outcome table (`OUTC26Q2.txt`) via a `LEFT JOIN` to prevent attrition of non-serious reports.
2. **Patient-Level Deduplication across Outcomes:** Account for the reality that a single patient report may record multiple outcome codes (e.g., hospitalization followed by death) by evaluating distinct cases per severity tier.
3. **Comparative Outcome Rates:** Compute proportional severity rates across clinical brands to evaluate whether obesity-indicated cohorts exhibit differential hospitalization or fatality profiles compared to diabetic populations.

In [ ]:
"""
FDA FAERS Pharmacovigilance Analytics Pipeline
Module: 03_glp1_clinical_outcomes.py
Author: Dalina Gaxiola Báez
Description:
    Quantifies and compares clinical severity outcomes (Death, Hospitalization,
    Life-Threatening events, Disability) across the clean GLP-1 monotherapy cohort.
    Enforces ICH E2D seriousness criteria and preserves non-serious cases using a LEFT JOIN.
"""

# Definición de la ruta hacia la tabla de desenlaces clínicos
OUTC_FILE = os.path.join(DATA_DIR, "OUTC26Q2.txt")

query_outcomes = rf"""
WITH
-- 1. Excluir casos descartados en logs oficiales de la FDA
retracted_cases AS (
    SELECT DISTINCT column0::VARCHAR AS caseid
    FROM read_csv('{DELETE_FILE}', header=False)
),

-- 2. Cohorte limpia de monoterapias GLP-1 por marca e indicación
clean_cohort AS (
    SELECT DISTINCT
        d.primaryid::VARCHAR AS primaryid,
        d.caseid::VARCHAR AS caseid,
        CASE
            WHEN UPPER(d.drugname) LIKE '%ZEPBOUND%' THEN 'ZEPBOUND (Tirzepatide - Obesity)'
            WHEN UPPER(d.drugname) LIKE '%MOUNJARO%' THEN 'MOUNJARO (Tirzepatide - T2D)'
            WHEN (UPPER(d.drugname) = 'TIRZEPATIDE' OR UPPER(d.prod_ai) = 'TIRZEPATIDE')
                 AND UPPER(d.drugname) NOT LIKE '%CYANOCOBALAMIN%'
                 AND UPPER(d.drugname) NOT LIKE '%PYRIDOXINE%'
                 AND UPPER(d.drugname) NOT LIKE '%GLYCINE%'
                 AND UPPER(d.drugname) NOT LIKE '%NIACINAMIDE%'
                 AND UPPER(d.drugname) NOT LIKE '%LEVOCARNITINE%'
                 AND UPPER(d.drugname) NOT LIKE '%METHYLCOBALAMIN%'
                 THEN 'TIRZEPATIDE (Generic/Unspecified)'

            WHEN UPPER(d.drugname) LIKE '%WEGOVY%' THEN 'WEGOVY (Semaglutide - Obesity)'
            WHEN UPPER(d.drugname) LIKE '%OZEMPIC%' THEN 'OZEMPIC (Semaglutide - T2D)'
            WHEN UPPER(d.drugname) LIKE '%RYBELSUS%' THEN 'RYBELSUS (Semaglutide - Oral T2D)'
            WHEN (UPPER(d.drugname) = 'SEMAGLUTIDE' OR UPPER(d.prod_ai) = 'SEMAGLUTIDE')
                 AND UPPER(d.drugname) NOT LIKE '%CYANOCOBALAMIN%'
                 AND UPPER(d.drugname) NOT LIKE '%PYRIDOXINE%'
                 AND UPPER(d.drugname) NOT LIKE '%GLYCINE%'
                 AND UPPER(d.drugname) NOT LIKE '%NIACINAMIDE%'
                 AND UPPER(d.drugname) NOT LIKE '%METHYLCOBALAMIN%'
                 THEN 'SEMAGLUTIDE (Generic/Unspecified)'

            WHEN UPPER(d.drugname) LIKE '%TRULICITY%' OR UPPER(d.prod_ai) = 'DULAGLUTIDE' THEN 'TRULICITY (Dulaglutide - T2D)'
            WHEN UPPER(d.drugname) LIKE '%SAXENDA%' THEN 'SAXENDA (Liraglutide - Obesity)'
            WHEN UPPER(d.drugname) LIKE '%VICTOZA%' THEN 'VICTOZA (Liraglutide - T2D)'
            WHEN (UPPER(d.drugname) = 'LIRAGLUTIDE' OR UPPER(d.prod_ai) = 'LIRAGLUTIDE')
                 AND UPPER(d.drugname) NOT LIKE '%INSULIN%'
                 AND UPPER(d.prod_ai) NOT LIKE '%INSULIN%'
                 THEN 'LIRAGLUTIDE (Generic/Unspecified)'

            WHEN UPPER(d.drugname) LIKE '%BYETTA%' OR UPPER(d.drugname) LIKE '%BYDUREON%'
                 OR UPPER(d.drugname) = 'EXENATIDE' OR UPPER(d.prod_ai) = 'EXENATIDE'
                 THEN 'EXENATIDE (Generic/Brand)'

            ELSE 'OTHER_OR_EXCLUDED'
        END AS clinical_brand_cohort,

        CASE
            WHEN LOWER(d.drugname) LIKE '%soliqua%'
                 OR LOWER(d.drugname) LIKE '%xultophy%'
                 OR LOWER(d.prod_ai) LIKE '%insulin%'
                 THEN 'EXCLUDE_INSULIN_COMBINATION'
            WHEN LOWER(d.drugname) LIKE '%cyanocobalamin%'
                 OR LOWER(d.drugname) LIKE '%pyridoxine%'
                 OR LOWER(d.drugname) LIKE '%glycine%'
                 OR LOWER(d.drugname) LIKE '%niacinamide%'
                 OR LOWER(d.drugname) LIKE '%levocarnitine%'
                 OR LOWER(d.drugname) LIKE '%methylcobalamin%'
                 OR LOWER(d.prod_ai) LIKE '%\\%'
                 THEN 'EXCLUDE_COMPOUNDED'
            ELSE 'KEEP_MONOTHERAPY'
        END AS exclusion_flag

    FROM read_csv('{DRUG_FILE}', delim='$', header=True) d
    WHERE d.role_cod = 'PS'
      AND d.caseid::VARCHAR NOT IN (SELECT caseid FROM retracted_cases)
),

filtered_cohort AS (
    SELECT DISTINCT primaryid, clinical_brand_cohort
    FROM clean_cohort
    WHERE exclusion_flag = 'KEEP_MONOTHERAPY'
      AND clinical_brand_cohort != 'OTHER_OR_EXCLUDED'
),

-- 3. Tabla de desenlaces clínicos (ICH E2D Seriousness Criteria)
outcomes AS (
    SELECT DISTINCT
        primaryid::VARCHAR AS primaryid,
        UPPER(TRIM(outc_cod)) AS outc_cod
    FROM read_csv('{OUTC_FILE}', delim='$', header=True)
)

-- 4. Métricas de severidad y tasas porcentuales por indicación y marca
SELECT
    c.clinical_brand_cohort,
    COUNT(DISTINCT c.primaryid) AS total_cases,

    -- Conteos absolutos por criterio de gravedad
    COUNT(DISTINCT CASE WHEN o.outc_cod = 'DE' THEN c.primaryid END) AS death_cases,
    COUNT(DISTINCT CASE WHEN o.outc_cod = 'LT' THEN c.primaryid END) AS life_threatening_cases,
    COUNT(DISTINCT CASE WHEN o.outc_cod = 'HO' THEN c.primaryid END) AS hospitalization_cases,
    COUNT(DISTINCT CASE WHEN o.outc_cod = 'DS' THEN c.primaryid END) AS disability_cases,

    -- Tasa de hospitalización (%)
    ROUND(
        COUNT(DISTINCT CASE WHEN o.outc_cod = 'HO' THEN c.primaryid END) * 100.0 /
        COUNT(DISTINCT c.primaryid), 2
    ) AS hospitalization_rate_pct,

    -- Tasa de casos graves críticos (% con Muerte, Riesgo Vital u Hospitalización)
    ROUND(
        COUNT(DISTINCT CASE WHEN o.outc_cod IN ('DE', 'LT', 'HO') THEN c.primaryid END) * 100.0 /
        COUNT(DISTINCT c.primaryid), 2
    ) AS serious_critical_rate_pct

FROM filtered_cohort c
LEFT JOIN outcomes o
    ON c.primaryid = o.primaryid
GROUP BY c.clinical_brand_cohort
ORDER BY total_cases DESC;
"""

# Ejecución y visualización
con = duckdb.connect()
df_severity_outcomes = con.execute(query_outcomes).df()

pd.set_option('display.max_rows', None)
display(df_severity_outcomes)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clinical_brand_cohort,total_cases,death_cases,life_threatening_cases,hospitalization_cases,disability_cases,hospitalization_rate_pct,serious_critical_rate_pct
0,ZEPBOUND (Tirzepatide - Obesity),13476,36,14,412,16,3.06,3.35
1,OZEMPIC (Semaglutide - T2D),8170,52,53,921,92,11.27,11.95
2,WEGOVY (Semaglutide - Obesity),6556,10,36,429,67,6.54,6.97
3,MOUNJARO (Tirzepatide - T2D),5202,57,97,810,92,15.57,17.26
4,TRULICITY (Dulaglutide - T2D),1004,20,12,214,7,21.31,23.21
5,SEMAGLUTIDE (Generic/Unspecified),584,4,13,66,8,11.30,12.84
6,RYBELSUS (Semaglutide - Oral T2D),439,6,4,76,16,17.31,18.68
7,TIRZEPATIDE (Generic/Unspecified),320,6,12,117,13,36.56,39.38
8,LIRAGLUTIDE (Generic/Unspecified),187,0,0,3,0,1.60,1.60
9,SAXENDA (Liraglutide - Obesity),95,2,5,22,7,23.16,26.32


## 4. Clinical Severity Profiling & Patient Outcomes Analysis

### 4.1 Evaluation of Seriousness Profiles across Clinical Cohorts
Cross-tabulating the harmonized monotherapy cohort with FAERS clinical outcome records (`OUTC26Q2.txt`) reveals marked disparities in seriousness profiles according to therapeutic indication:

| Clinical Cohort | Target Indication | Total Cases | Hospitalization Rate (%) | Critical Serious Rate (%) | Data-Driven Finding |
|---|---|:---:|:---:|:---:|---|
| **Zepbound** | Obesity | 13,476 | 3.06% | 3.35% | Lowest critical seriousness and hospitalization proportion across high-volume brands in the quarterly extract. |
| **Mounjaro** | Type 2 Diabetes | 5,202 | 15.57% | 17.26% | Over a 5-fold increase in critical outcome rate compared to *Zepbound*, reflecting higher severity reporting in diabetic populations. |
| **Wegovy** | Obesity | 6,556 | 6.54% | 6.97% | Demonstrates moderate hospitalization and critical seriousness rates, consistently lower than diabetes-indicated Semaglutide. |
| **Ozempic** | Type 2 Diabetes | 8,170 | 11.27% | 11.95% | Nearly double the critical seriousness rate of *Wegovy*, confirming the clinical disparity between diabetic and non-diabetic cohorts. |
| **Trulicity** | Type 2 Diabetes | 1,004 | 21.31% | 23.21% | High proportional hospitalization rate among established monotherapies. |

*Note: Critical Serious Rate includes all reports resulting in Death (DE), Life-Threatening experience (LT), or Inpatient Hospitalization (HO) per ICH E2D criteria.*

---

### 4.2 Next Step: MedDRA Preferred Term (PT) Adverse Reaction Profiling (Query 04)
Having identified that diabetic cohorts experience higher hospitalization rates, the next analytical objective is to determine the specific reported clinical symptoms and diagnoses without prior assumptions.

#### Methodological Objectives for Module 04 (`04_glp1_adverse_reactions.py`):
1. **Relational Integration with Adverse Reactions (`REAC26Q2.txt`):** Link the clean cohort against the reaction table using standardized Medical Dictionary for Regulatory Activities (MedDRA) Preferred Terms (`PT`).
2. **Top Adverse Drug Reaction (ADR) Mapping:** Quantify the top 10 most frequently reported clinical symptoms and diagnoses for each major brand to empirically identify which adverse reactions predominate in each cohort.
3. **Safety Signal Screening:** Compare whether obesity and diabetes cohorts differ in their actual reported reaction profiles (e.g., gastrointestinal events, metabolic conditions, or administration site issues).

In [ ]:
"""
FDA FAERS Pharmacovigilance Analytics Pipeline
Module: 04_glp1_adverse_reactions.py
Author: Dalina Gaxiola Báez
Description:
    Integrates the clean GLP-1 monotherapy cohort with MedDRA Preferred Terms (PT)
    from REAC26Q2.txt. Ranks the top 5 most frequently reported adverse drug reactions
    for each major brand cohort and computes patient-level reporting prevalence rates.
"""

# Definición de la ruta hacia la tabla de reacciones adversas
REAC_FILE = os.path.join(DATA_DIR, "REAC26Q2.txt")

query_top_reactions = rf"""
WITH
-- 1. Excluir casos descartados por la FDA
retracted_cases AS (
    SELECT DISTINCT column0::VARCHAR AS caseid
    FROM read_csv('{DELETE_FILE}', header=False)
),

-- 2. Cohorte limpia de monoterapias GLP-1 por marca e indicación
clean_cohort AS (
    SELECT DISTINCT
        d.primaryid::VARCHAR AS primaryid,
        d.caseid::VARCHAR AS caseid,
        CASE
            WHEN UPPER(d.drugname) LIKE '%ZEPBOUND%' THEN 'ZEPBOUND (Tirzepatide - Obesity)'
            WHEN UPPER(d.drugname) LIKE '%MOUNJARO%' THEN 'MOUNJARO (Tirzepatide - T2D)'
            WHEN UPPER(d.drugname) LIKE '%WEGOVY%' THEN 'WEGOVY (Semaglutide - Obesity)'
            WHEN UPPER(d.drugname) LIKE '%OZEMPIC%' THEN 'OZEMPIC (Semaglutide - T2D)'
            ELSE 'OTHER'
        END AS clinical_brand_cohort,

        CASE
            WHEN LOWER(d.drugname) LIKE '%soliqua%'
                 OR LOWER(d.drugname) LIKE '%xultophy%'
                 OR LOWER(d.prod_ai) LIKE '%insulin%'
                 THEN 'EXCLUDE_INSULIN_COMBINATION'
            WHEN LOWER(d.drugname) LIKE '%cyanocobalamin%'
                 OR LOWER(d.drugname) LIKE '%pyridoxine%'
                 OR LOWER(d.drugname) LIKE '%glycine%'
                 OR LOWER(d.drugname) LIKE '%niacinamide%'
                 OR LOWER(d.drugname) LIKE '%levocarnitine%'
                 OR LOWER(d.drugname) LIKE '%methylcobalamin%'
                 OR LOWER(d.prod_ai) LIKE '%\\%'
                 THEN 'EXCLUDE_COMPOUNDED'
            ELSE 'KEEP_MONOTHERAPY'
        END AS exclusion_flag

    FROM read_csv('{DRUG_FILE}', delim='$', header=True) d
    WHERE d.role_cod = 'PS'
      AND d.caseid::VARCHAR NOT IN (SELECT caseid FROM retracted_cases)
),

target_cohort AS (
    SELECT DISTINCT primaryid, clinical_brand_cohort
    FROM clean_cohort
    WHERE exclusion_flag = 'KEEP_MONOTHERAPY'
      AND clinical_brand_cohort IN (
          'ZEPBOUND (Tirzepatide - Obesity)',
          'MOUNJARO (Tirzepatide - T2D)',
          'WEGOVY (Semaglutide - Obesity)',
          'OZEMPIC (Semaglutide - T2D)'
      )
),

-- Denominador total de pacientes por cohorte para calcular prevalencias exactas
cohort_denominators AS (
    SELECT
        clinical_brand_cohort,
        COUNT(DISTINCT primaryid) AS cohort_total_cases
    FROM target_cohort
    GROUP BY clinical_brand_cohort
),

-- 3. Cargar tabla de Reacciones MedDRA Preferred Terms (PT)
reactions AS (
    SELECT DISTINCT
        primaryid::VARCHAR AS primaryid,
        UPPER(TRIM(pt)) AS meddra_preferred_term
    FROM read_csv('{REAC_FILE}', delim='$', header=True)
),

-- 4. Conteo y ranking de reacciones por cada cohorte
ranked_reactions AS (
    SELECT
        t.clinical_brand_cohort,
        r.meddra_preferred_term,
        COUNT(DISTINCT t.primaryid) AS patient_cases_with_reaction,
        cd.cohort_total_cases,
        ROUND(
            COUNT(DISTINCT t.primaryid) * 100.0 / cd.cohort_total_cases, 2
        ) AS reaction_prevalence_pct,
        ROW_NUMBER() OVER (
            PARTITION BY t.clinical_brand_cohort
            ORDER BY COUNT(DISTINCT t.primaryid) DESC
        ) AS reaction_rank
    FROM target_cohort t
    JOIN reactions r
        ON t.primaryid = r.primaryid
    JOIN cohort_denominators cd
        ON t.clinical_brand_cohort = cd.clinical_brand_cohort
    GROUP BY
        t.clinical_brand_cohort,
        r.meddra_preferred_term,
        cd.cohort_total_cases
)

-- 5. Extraer el Top 5 empírico de reacciones para cada fármaco
SELECT
    clinical_brand_cohort,
    reaction_rank,
    meddra_preferred_term,
    patient_cases_with_reaction,
    cohort_total_cases,
    reaction_prevalence_pct
FROM ranked_reactions
WHERE reaction_rank <= 5
ORDER BY clinical_brand_cohort, reaction_rank;
"""

# Ejecutar y visualizar
con = duckdb.connect()
df_top_reactions = con.execute(query_top_reactions).df()

pd.set_option('display.max_rows', None)
display(df_top_reactions)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clinical_brand_cohort,reaction_rank,meddra_preferred_term,patient_cases_with_reaction,cohort_total_cases,reaction_prevalence_pct
0,MOUNJARO (Tirzepatide - T2D),1,NAUSEA,584,5202,11.23
1,MOUNJARO (Tirzepatide - T2D),2,DIARRHOEA,465,5202,8.94
2,MOUNJARO (Tirzepatide - T2D),3,VOMITING,443,5202,8.52
3,MOUNJARO (Tirzepatide - T2D),4,CONSTIPATION,300,5202,5.77
4,MOUNJARO (Tirzepatide - T2D),5,ABDOMINAL PAIN,248,5202,4.77
5,OZEMPIC (Semaglutide - T2D),1,OFF LABEL USE,998,8170,12.22
6,OZEMPIC (Semaglutide - T2D),2,NAUSEA,967,8170,11.84
7,OZEMPIC (Semaglutide - T2D),3,IMPAIRED GASTRIC EMPTYING,746,8170,9.13
8,OZEMPIC (Semaglutide - T2D),4,WEIGHT DECREASED,634,8170,7.76
9,OZEMPIC (Semaglutide - T2D),5,WEIGHT INCREASED,624,8170,7.64


## 5. Granular Adverse Reaction Profiling: Top 15 MedDRA PTs & Taxonomy Stratification

### 5.1 Rationale for Methodological Expansion
Our initial exploratory evaluation of the top 5 MedDRA Preferred Terms (PTs) revealed a substantial presence of operational friction, drug administration errors, and secular reporting artifacts:
* Terms such as `OFF LABEL USE` (*Ozempic* #1), `INCORRECT DOSE ADMINISTERED` (*Zepbound* #2), `DEVICE LEAKAGE` (*Wegovy* #4), and lack of therapeutic efficacy terms (`DRUG INEFFECTIVE`, `WEIGHT LOSS POOR`, `WEIGHT INCREASED`) occupied primary positions in spontaneous report frequencies.
* Restricting analysis to the top 5 artificially obscured lower-frequency biological toxicities and physiological reactions, constraining the clinical evaluation mandated by our study objectives.

To establish an exhaustive, clinically validated safety profile without discarding real-world evidence on drug administration, we implement two methodological improvements:
1. **Expansion to Top 15 Preferred Terms:** Broadens the analytical window across each of the four primary cohorts (*Zepbound*, *Mounjaro*, *Wegovy*, *Ozempic*) to capture underlying systemic, gastrointestinal, and metabolic symptoms.
2. **Clinical Taxonomy Stratification (`event_category`):** In accordance with FDA Combination Product surveillance guidelines, each MedDRA term is systematically mapped into one of two operational categories:
   * **`CLINICAL_ADVERSE_REACTION`:** Genuine biological symptoms, pathophysiological states, and clinical toxicities (e.g., *Nausea, Vomiting, Impaired Gastric Emptying, Diarrhoea, Constipation, Abdominal Pain, Pancreatitis, Hypoglycemia*).
   * **`OPERATIONAL_OR_USE_ISSUE`:** Device engineering failures, user administration errors, therapeutic non-response, and off-label usage patterns (e.g., *Device Leakage, Incorrect Dose Administered, Drug Ineffective, Off Label Use, Product Use Issue*).

---

### 5.2 Next Step: Multi-Cohort Reaction Triage (Query 04 Extended)
Module 04 (`04_glp1_adverse_reactions_top15_stratified.py`) executes this taxonomy layer across our deduplicated monotherapy cohorts, computing patient-level reporting prevalence rates for the top 15 terms per brand.

In [ ]:
"""
FDA FAERS Pharmacovigilance Analytics Pipeline
Module: 04_glp1_adverse_reactions_top15_stratified.py
Author: Dalina Gaxiola Báez
Description:
    Expands MedDRA Preferred Term (PT) analysis to the Top 15 most frequent reported events
    per major brand cohort. Implements a clinical taxonomy layer to formally differentiate:
    1) CLINICAL_ADVERSE_REACTION (Biological symptoms, physiological toxicities, pathology)
    2) OPERATIONAL_AND_USE_ISSUE (Device malfunctions, dosing errors, off-label use, lack of efficacy)
"""

query_top15_stratified = rf"""
WITH
-- 1. Excluir casos revocados en logs oficiales de la FDA
retracted_cases AS (
    SELECT DISTINCT column0::VARCHAR AS caseid
    FROM read_csv('{DELETE_FILE}', header=False)
),

-- 2. Cohorte limpia de monoterapias GLP-1
clean_cohort AS (
    SELECT DISTINCT
        d.primaryid::VARCHAR AS primaryid,
        d.caseid::VARCHAR AS caseid,
        CASE
            WHEN UPPER(d.drugname) LIKE '%ZEPBOUND%' THEN 'ZEPBOUND (Tirzepatide - Obesity)'
            WHEN UPPER(d.drugname) LIKE '%MOUNJARO%' THEN 'MOUNJARO (Tirzepatide - T2D)'
            WHEN UPPER(d.drugname) LIKE '%WEGOVY%' THEN 'WEGOVY (Semaglutide - Obesity)'
            WHEN UPPER(d.drugname) LIKE '%OZEMPIC%' THEN 'OZEMPIC (Semaglutide - T2D)'
            ELSE 'OTHER'
        END AS clinical_brand_cohort,

        CASE
            WHEN LOWER(d.drugname) LIKE '%soliqua%'
                 OR LOWER(d.drugname) LIKE '%xultophy%'
                 OR LOWER(d.prod_ai) LIKE '%insulin%'
                 THEN 'EXCLUDE_INSULIN_COMBINATION'
            WHEN LOWER(d.drugname) LIKE '%cyanocobalamin%'
                 OR LOWER(d.drugname) LIKE '%pyridoxine%'
                 OR LOWER(d.drugname) LIKE '%glycine%'
                 OR LOWER(d.drugname) LIKE '%niacinamide%'
                 OR LOWER(d.drugname) LIKE '%levocarnitine%'
                 OR LOWER(d.drugname) LIKE '%methylcobalamin%'
                 OR LOWER(d.prod_ai) LIKE '%\\%'
                 THEN 'EXCLUDE_COMPOUNDED'
            ELSE 'KEEP_MONOTHERAPY'
        END AS exclusion_flag

    FROM read_csv('{DRUG_FILE}', delim='$', header=True) d
    WHERE d.role_cod = 'PS'
      AND d.caseid::VARCHAR NOT IN (SELECT caseid FROM retracted_cases)
),

target_cohort AS (
    SELECT DISTINCT primaryid, clinical_brand_cohort
    FROM clean_cohort
    WHERE exclusion_flag = 'KEEP_MONOTHERAPY'
      AND clinical_brand_cohort IN (
          'ZEPBOUND (Tirzepatide - Obesity)',
          'MOUNJARO (Tirzepatide - T2D)',
          'WEGOVY (Semaglutide - Obesity)',
          'OZEMPIC (Semaglutide - T2D)'
      )
),

-- Denominador exacto por marca para prevalencias relativas
cohort_denominators AS (
    SELECT
        clinical_brand_cohort,
        COUNT(DISTINCT primaryid) AS cohort_total_cases
    FROM target_cohort
    GROUP BY clinical_brand_cohort
),

-- 3. Cargar tabla de Reacciones MedDRA PT
reactions AS (
    SELECT DISTINCT
        primaryid::VARCHAR AS primaryid,
        UPPER(TRIM(pt)) AS meddra_preferred_term
    FROM read_csv('{REAC_FILE}', delim='$', header=True)
),

-- 4. Conteo, clasificación taxonómica y ranking Top 15
ranked_reactions AS (
    SELECT
        t.clinical_brand_cohort,
        r.meddra_preferred_term,

        -- Taxonomía clínica reguladora: Biológico vs. Operativo/Uso
        CASE
            WHEN r.meddra_preferred_term IN (
                'OFF LABEL USE',
                'INCORRECT DOSE ADMINISTERED',
                'DRUG INEFFECTIVE',
                'WEIGHT LOSS POOR',
                'WEIGHT INCREASED',
                'DEVICE LEAKAGE',
                'PRODUCT USE ISSUE',
                'INAPPROPRIATE SCHEDULE OF PRODUCT ADMINISTRATION',
                'WRONG TECHNIQUE IN PRODUCT USAGE PROCESS',
                'DEVICE DEFECTIVE',
                'UNDERDOSE',
                'OVERDOSE',
                'ACCIDENTAL OVERDOSE',
                'PRODUCT COMPLAINT',
                'EXPIRED PRODUCT ADMINISTERED'
            ) THEN 'OPERATIONAL_OR_USE_ISSUE'
            ELSE 'CLINICAL_ADVERSE_REACTION'
        END AS event_category,

        COUNT(DISTINCT t.primaryid) AS patient_cases_with_reaction,
        cd.cohort_total_cases,
        ROUND(
            COUNT(DISTINCT t.primaryid) * 100.0 / cd.cohort_total_cases, 2
        ) AS reaction_prevalence_pct,
        ROW_NUMBER() OVER (
            PARTITION BY t.clinical_brand_cohort
            ORDER BY COUNT(DISTINCT t.primaryid) DESC
        ) AS reaction_rank
    FROM target_cohort t
    JOIN reactions r
        ON t.primaryid = r.primaryid
    JOIN cohort_denominators cd
        ON t.clinical_brand_cohort = cd.clinical_brand_cohort
    GROUP BY
        t.clinical_brand_cohort,
        r.meddra_preferred_term,
        cd.cohort_total_cases
)

-- 5. Extraer el Top 15 empírico para cada cohorte
SELECT
    clinical_brand_cohort,
    reaction_rank,
    meddra_preferred_term,
    event_category,
    patient_cases_with_reaction,
    cohort_total_cases,
    reaction_prevalence_pct
FROM ranked_reactions
WHERE reaction_rank <= 15
ORDER BY clinical_brand_cohort, reaction_rank;
"""

# Ejecutar y mostrar sin truncar
con = duckdb.connect()
df_top15_reactions = con.execute(query_top15_stratified).df()

pd.set_option('display.max_rows', None)
display(df_top15_reactions)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clinical_brand_cohort,reaction_rank,meddra_preferred_term,event_category,patient_cases_with_reaction,cohort_total_cases,reaction_prevalence_pct
0,MOUNJARO (Tirzepatide - T2D),1,NAUSEA,CLINICAL_ADVERSE_REACTION,584,5202,11.23
1,MOUNJARO (Tirzepatide - T2D),2,DIARRHOEA,CLINICAL_ADVERSE_REACTION,465,5202,8.94
2,MOUNJARO (Tirzepatide - T2D),3,VOMITING,CLINICAL_ADVERSE_REACTION,443,5202,8.52
3,MOUNJARO (Tirzepatide - T2D),4,CONSTIPATION,CLINICAL_ADVERSE_REACTION,300,5202,5.77
4,MOUNJARO (Tirzepatide - T2D),5,ABDOMINAL PAIN,CLINICAL_ADVERSE_REACTION,248,5202,4.77
5,MOUNJARO (Tirzepatide - T2D),6,INCORRECT DOSE ADMINISTERED,OPERATIONAL_OR_USE_ISSUE,242,5202,4.65
6,MOUNJARO (Tirzepatide - T2D),7,OFF LABEL USE,OPERATIONAL_OR_USE_ISSUE,229,5202,4.40
7,MOUNJARO (Tirzepatide - T2D),8,INJECTION SITE PAIN,CLINICAL_ADVERSE_REACTION,209,5202,4.02
8,MOUNJARO (Tirzepatide - T2D),9,DRUG INEFFECTIVE,OPERATIONAL_OR_USE_ISSUE,195,5202,3.75
9,MOUNJARO (Tirzepatide - T2D),10,IMPAIRED GASTRIC EMPTYING,CLINICAL_ADVERSE_REACTION,181,5202,3.48


## 7. Exhaustive Adverse Event Landscape & Taxonomy Audit

### 7.1 Methodological Rationale: Eliminating Arbitrary Rank Truncation
Restricting adverse event surveillance to arbitrary thresholds (such as Top 5 or Top 15) introduces severe truncation bias. While high-frequency, non-fatal symptoms (nausea, vomiting) dominate rank orders, high-severity clinical signals (e.g., pancreatitis, intestinal obstruction/ileus, thyroid neoplasms, or ketoacidosis) naturally exhibit lower absolute incidence and reside within the long-tail distribution of spontaneous reporting.

To eliminate truncation bias and conduct a complete pharmacovigilance audit, this module evaluates the entire spectrum of reported MedDRA Preferred Terms (PTs) across the four primary cohorts (*Zepbound*, *Mounjaro*, *Wegovy*, *Ozempic*):

1. **Macro Distribution Analysis:** Quantifies the global proportion of clinical biological reactions versus device malfunctions, dosing errors, and therapeutic efficacy complaints across the full event universe.
2. **Exhaustive Term Extraction:** Aggregates every unique MedDRA PT reported in 2026 Q2 without record limits, establishing the full baseline required for targeted safety queries and formal disproportionality scoring (PRR/ROR).

## 8. Exhaustive Safety & Critical Outcome Triage: ZEPBOUND (Tirzepatide - Obesity)

### 8.1 Methodological Rationale: Eliminating Truncation & Frequency Biases
Standard pharmacovigilance workflows frequently suffer from **frequency bias** and **arbitrary truncation** (e.g., evaluating only the Top 5 or Top 30 reported terms). In spontaneous reporting datasets:
* High-frequency, mild-to-moderate symptoms (such as nausea, diarrhea, or localized injection site discomfort) occupy top percentile ranks.
* Severe, low-incidence clinical toxicities that justify regulatory warnings (e.g., pancreatitis, acute kidney failure, intestinal obstruction, or fatal outcomes) remain hidden across the long-tail distribution of adverse events.

To ensure zero loss of critical safety signals, this section executes an unconstrained, two-tier epidemiological audit for **ZEPBOUND** (13,476 unique patient cases):

1. **Global Landscape & Administrative Friction:** Audits the entire unconstrained inventory of 1,314 unique MedDRA terms to establish the macro-level distribution between biological pathologies, autoinjector device malfunctions, medication errors, and therapeutic failures.
2. **Exhaustive Critical Outcome Stratification (ICH E2D):** Cross-tabulates every reported term specifically against cases resulting in **Death (`DE`)**, **Life-Threatening experiences (`LT`)**, or **Inpatient Hospitalization (`HO`)**. This isolates the exact medical diagnoses and complications driving serious clinical deterioration without discarding lower-frequency signals.

### 8.2 Execution Plan: Module 05_zepbound_exhaustive_safety.py
In the following pipeline execution, we compute:
* The macro-regulatory category distribution across all reported mentions.
* The complete distribution of clinical terms associated with serious critical outcomes (`DE`, `LT`, `HO`), reporting exact case counts for deaths, life-threatening events, and inpatient hospitalizations.


In [ ]:
"""
FDA FAERS Pharmacovigilance Analytics Pipeline
Module: 05_zepbound_exhaustive_safety.py
Author: Dalina Gaxiola Báez
Description:
    Conducts an unconstrained safety and critical outcome audit for ZEPBOUND (Tirzepatide - Obesity).
    1) Summarizes the macro-regulatory classification across all reported events.
    2) Extracts all MedDRA Preferred Terms (PTs) specifically linked to critical outcomes
       per ICH E2D criteria: Death (DE), Life-Threatening (LT), and Hospitalization (HO).
"""

import os
import duckdb
import pandas as pd

# ==========================================
# 1. ENVIRONMENT & PATH CONFIGURATION
# ==========================================
DATA_DIR = "/content/drive/MyDrive/TripleTen Projects/Proyectos/FAERS"
DELETE_FILE = os.path.join(DATA_DIR, "DELETE26Q2.txt")
DRUG_FILE = os.path.join(DATA_DIR, "DRUG26Q2.txt")
REAC_FILE = os.path.join(DATA_DIR, "REAC26Q2.txt")
OUTC_FILE = os.path.join(DATA_DIR, "OUTC26Q2.txt")

# ==========================================
# 2. EXHAUSTIVE PIPELINE QUERY
# ==========================================
query_zepbound_complete = rf"""
WITH
-- 1. Purge officially retracted cases per FDA maintenance logs
retracted_cases AS (
    SELECT DISTINCT column0::VARCHAR AS caseid
    FROM read_csv('{DELETE_FILE}', header=False)
),

-- 2. Target Clean Zepbound Monotherapy Cohort
zepbound_cohort AS (
    SELECT DISTINCT
        d.primaryid::VARCHAR AS primaryid
    FROM read_csv('{DRUG_FILE}', delim='$', header=True) d
    WHERE d.role_cod = 'PS'
      AND UPPER(d.drugname) LIKE '%ZEPBOUND%'
      AND d.caseid::VARCHAR NOT IN (SELECT caseid FROM retracted_cases)
      AND LOWER(d.drugname) NOT LIKE '%soliqua%'
      AND LOWER(d.drugname) NOT LIKE '%xultophy%'
      AND LOWER(d.drugname) NOT LIKE '%cyanocobalamin%'
      AND LOWER(d.drugname) NOT LIKE '%pyridoxine%'
      AND LOWER(d.drugname) NOT LIKE '%glycine%'
      AND LOWER(d.drugname) NOT LIKE '%niacinamide%'
      AND LOWER(d.drugname) NOT LIKE '%levocarnitine%'
      AND LOWER(d.drugname) NOT LIKE '%methylcobalamin%'
),

-- 3. Critical Clinical Outcomes (ICH E2D Criteria: DE, LT, HO)
critical_outcomes AS (
    SELECT DISTINCT
        primaryid::VARCHAR AS primaryid,
        UPPER(TRIM(outc_cod)) AS outc_cod
    FROM read_csv('{OUTC_FILE}', delim='$', header=True)
    WHERE UPPER(TRIM(outc_cod)) IN ('DE', 'LT', 'HO')
),

-- Patient-level severity classification
patient_severity AS (
    SELECT
        z.primaryid,
        MAX(CASE WHEN o.outc_cod = 'DE' THEN 1 ELSE 0 END) AS is_death,
        MAX(CASE WHEN o.outc_cod = 'LT' THEN 1 ELSE 0 END) AS is_life_threatening,
        MAX(CASE WHEN o.outc_cod = 'HO' THEN 1 ELSE 0 END) AS is_hospitalized,
        MAX(CASE WHEN o.outc_cod IS NOT NULL THEN 1 ELSE 0 END) AS is_critical_case
    FROM zepbound_cohort z
    LEFT JOIN critical_outcomes o
        ON z.primaryid = o.primaryid
    GROUP BY z.primaryid
),

-- 4. MedDRA Preferred Terms Extraction
reactions AS (
    SELECT DISTINCT
        primaryid::VARCHAR AS primaryid,
        UPPER(TRIM(pt)) AS meddra_preferred_term
    FROM read_csv('{REAC_FILE}', delim='$', header=True)
),

-- 5. Unified Patient-Reaction Mapping & Regulatory Taxonomy Layer
patient_reactions AS (
    SELECT
        ps.primaryid,
        ps.is_death,
        ps.is_life_threatening,
        ps.is_hospitalized,
        ps.is_critical_case,
        r.meddra_preferred_term,

        -- Regulatory taxonomy mapping
        CASE
            WHEN r.meddra_preferred_term LIKE '%DEVICE%'
                 OR r.meddra_preferred_term LIKE '%NEEDLE%'
                 OR r.meddra_preferred_term LIKE '%SYRINGE%'
                 OR r.meddra_preferred_term LIKE '%PRODUCT QUALITY%'
                 OR r.meddra_preferred_term = 'PRODUCT COMPLAINT'
                 THEN 'DEVICE_OR_PRODUCT_QUALITY'

            WHEN r.meddra_preferred_term LIKE '%DOSE%'
                 OR r.meddra_preferred_term LIKE '%OVERDOSE%'
                 OR r.meddra_preferred_term LIKE '%UNDERDOSE%'
                 OR r.meddra_preferred_term LIKE '%ADMINIST%'
                 OR r.meddra_preferred_term LIKE '%WRONG%'
                 OR r.meddra_preferred_term LIKE '%SCHEDULE%'
                 OR r.meddra_preferred_term = 'PRODUCT USE ISSUE'
                 THEN 'MEDICATION_ADMINISTRATION_ERROR'

            WHEN r.meddra_preferred_term LIKE '%OFF LABEL%'
                 OR r.meddra_preferred_term LIKE '%UNAPPROVED INDICATION%'
                 THEN 'OFF_LABEL_USAGE'

            WHEN r.meddra_preferred_term IN (
                'DRUG INEFFECTIVE', 'WEIGHT LOSS POOR', 'WEIGHT INCREASED',
                'THERAPEUTIC RESPONSE CHANGED', 'TREATMENT FAILURE',
                'THERAPEUTIC PRODUCT INEFFECTIVE'
            ) THEN 'THERAPEUTIC_EFFICACY_ISSUE'

            ELSE 'CLINICAL_ADVERSE_REACTION'
        END AS regulatory_category

    FROM patient_severity ps
    JOIN reactions r
        ON ps.primaryid = r.primaryid
)

SELECT * FROM patient_reactions;
"""

# ==========================================
# 3. PIPELINE EXECUTION & DATA STRUCTURES
# ==========================================
con = duckdb.connect()
df_zepbound_all_events = con.execute(query_zepbound_complete).df()

print(f"Total event-patient records retrieved for Zepbound: {len(df_zepbound_all_events)}")
print(f"Unique patient cases evaluated: {df_zepbound_all_events['primaryid'].nunique()}")

# ----------------------------------------------------
# TABLE A: GLOBAL MACRO-REGULATORY DISTRIBUTION
# ----------------------------------------------------
print("\n" + "="*80)
print("TABLE A: MACRO-REGULATORY DISTRIBUTION (ZEPBOUND COMPLETE UNIVERSE)")
print("="*80)

macro_zepbound = df_zepbound_all_events.groupby('regulatory_category').agg(
    total_mentions=('primaryid', 'count'),
    unique_meddra_terms=('meddra_preferred_term', 'nunique'),
    unique_patients=('primaryid', 'nunique')
).reset_index()

macro_zepbound['pct_mentions'] = (
    macro_zepbound['total_mentions'] * 100.0 / macro_zepbound['total_mentions'].sum()
).round(2)

macro_zepbound = macro_zepbound.sort_values(by='total_mentions', ascending=False)
display(macro_zepbound)

# ----------------------------------------------------
# TABLE B: EXHAUSTIVE CRITICAL OUTCOME TRIAGE (DE, LT, HO)
# ----------------------------------------------------
print("\n" + "="*80)
print("TABLE B: REPORTED EVENTS IN CRITICAL CASES (DEATH, LIFE-THREATENING, HOSPITALIZATION)")
print("="*80)

# Filter exclusively for patients presenting critical outcomes per ICH E2D
df_critical_only = df_zepbound_all_events[df_zepbound_all_events['is_critical_case'] == 1]

critical_terms_summary = df_critical_only.groupby(
    ['meddra_preferred_term', 'regulatory_category']
).agg(
    total_critical_cases=('primaryid', 'nunique'),
    deaths=('is_death', 'sum'),
    life_threatening=('is_life_threatening', 'sum'),
    hospitalizations=('is_hospitalized', 'sum')
).reset_index()

critical_terms_summary = critical_terms_summary.sort_values(by='total_critical_cases', ascending=False)

print(f"Total unique MedDRA terms documented across critical cases: {len(critical_terms_summary)}")

# Display top 50 critical terms for immediate review
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', None)
display(critical_terms_summary.head(50))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total event-patient records retrieved for Zepbound: 27532
Unique patient cases evaluated: 13476

TABLE A: MACRO-REGULATORY DISTRIBUTION (ZEPBOUND COMPLETE UNIVERSE)


,regulatory_category,total_mentions,unique_meddra_terms,unique_patients,pct_mentions
0,CLINICAL_ADVERSE_REACTION,22206,1260,10656,80.66
2,MEDICATION_ADMINISTRATION_ERROR,2607,31,2438,9.47
4,THERAPEUTIC_EFFICACY_ISSUE,2320,5,2127,8.43
3,OFF_LABEL_USAGE,310,2,310,1.13
1,DEVICE_OR_PRODUCT_QUALITY,89,16,83,0.32



TABLE B: REPORTED EVENTS IN CRITICAL CASES (DEATH, LIFE-THREATENING, HOSPITALIZATION)
Total unique MedDRA terms documented across critical cases: 428


,meddra_preferred_term,regulatory_category,total_critical_cases,deaths,life_threatening,hospitalizations
418,VOMITING,CLINICAL_ADVERSE_REACTION,69,0,2,69
284,NAUSEA,CLINICAL_ADVERSE_REACTION,52,0,1,51
102,DEHYDRATION,CLINICAL_ADVERSE_REACTION,50,1,0,50
106,DIARRHOEA,CLINICAL_ADVERSE_REACTION,45,2,1,44
89,CONSTIPATION,CLINICAL_ADVERSE_REACTION,27,0,0,27
2,ABDOMINAL PAIN,CLINICAL_ADVERSE_REACTION,22,0,0,22
98,DEATH,CLINICAL_ADVERSE_REACTION,22,22,0,1
310,PANCREATITIS,CLINICAL_ADVERSE_REACTION,20,0,0,20
79,CHOLELITHIASIS,CLINICAL_ADVERSE_REACTION,18,1,0,17
3,ABDOMINAL PAIN UPPER,CLINICAL_ADVERSE_REACTION,17,0,0,17


## 8.3 Analytical Findings: Dual-Tier Safety Evaluation of Zepbound

### 1. Macro-Level Evidence (The Complete 13,476 Patient Landscape)
* **Biological Preponderance:** 80.66% of all recorded adverse events (22,206 mentions across 1,260 unique MedDRA terms) represent genuine clinical adverse drug reactions.
* **Administrative & Mechanical Friction:** Over 17.9% of reports do not reflect pharmacological toxicities, but rather medication errors (9.47%, primarily dosing confusion) and therapeutic expectations/lack of efficacy (8.43%).
* **Minimal Off-Label Contamination:** Unlike diabetes-indicated incretins, off-label usage represents only 1.13% of reports, confirming clear indication alignment with obesity management.

### 2. Micro-Level Evidence (Exhaustive Critical Outcome Audit)
Filtering strictly for ICH E2D serious clinical outcomes (Death: 36, Life-Threatening: 14, Hospitalization: 412) uncovers 428 distinct MedDRA terms driving acute medical escalation:
* **Gastrointestinal & Biliary Emergencies:**
  * Acute pancreatitis (`PANCREATITIS`: 20 cases; `PANCREATITIS ACUTE`: 6 cases) and gallbladder pathologies (`CHOLELITHIASIS`: 18 cases, 1 death; `GALLBLADDER DISORDER`: 9 cases) account for a significant share of inpatient admissions.
  * Mechanical and dysmotility crises (`INTESTINAL OBSTRUCTION`: 9 cases; `IMPAIRED GASTRIC EMPTYING`: 10 cases; `COLITIS ISCHAEMIC`: 6 cases) confirm intestinal hypomotility risks.
* **Volume Depletion Cascade:** Severe fluid loss (`DEHYDRATION`: 50 hospitalizations, 1 death) secondary to intractable vomiting (69 hospitalizations) represents the primary metabolic pathway to emergency admission.
* **Immediate Systemic & Vascular Toxicities:** Severe immune-mediated emergencies (`ANAPHYLACTIC SHOCK`: 7 cases; `ANAPHYLACTIC REACTION`: 6 cases) and acute cardiovascular/cerebrovascular events (`ATRIAL FIBRILLATION`: 10 cases, 1 death; `CEREBROVASCULAR ACCIDENT`: 6 cases; `SEPSIS`: 12 cases, 2 deaths) delineate the definitive fatal and life-threatening spectrum.